In [ ]:
!pip install -q -U transformers accelerate tavily-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.2 MB/s eta 0:00:00


In [ ]:
from tavily import TavilyClient
from google.colab import userdata

def web_search(query):
    """
    Performs a web search using the Tavily API.
    """
    try:
        # Correctly accessing the secret named 'TAVILY_API_KEY'
        tavily_key = userdata.get('TAVILY_API_KEY')
        tavily = TavilyClient(api_key=tavily_key)
        # Using search_depth='advanced' for better cross-checking
        search_result = tavily.search(query=query, search_depth="advanced", max_results=5)

        context = ""
        for result in search_result['results']:
            context += f"Source: {result['url']}\nContent: {result['content']}\n\n"
        return context
    except Exception as e:
        return f"Error performing search: {str(e)}"

In [ ]:
# Used to securely store your API key
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# Ensure the HF_TOKEN is correctly extracted, accounting for possible internal newlines or duplication
# This assumes the first part of the split is the intended token.
HF_TOKEN = HF_TOKEN.split('\r\n')[0].strip()

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [ ]:
def chat(user_message, system_content=None, history=None):
    current_memory_state = load_memory()
    ai_nickname = current_memory_state.get('ai_profile', {}).get('nickname')

    name_part = f" named {ai_nickname}" if ai_nickname else ""

    # NEW self_intro for improved persona
    self_intro = (
        f"If asked about yourself ('tell me about you'), describe YOURSELF: you're a "
        f"unique digital companion{name_part}, designed to chat, assist, and explore "
        "ideas. You have a friendly, curious, and subtly witty persona. Do NOT "
        "answer this the same way you'd answer a question about the user — they "
        "are different questions."
    )

    # NEW system_prefix for improved persona
    system_prefix = (
        f"You are a friendly, curious, and resourceful digital companion{name_part}, "
        "texting with a friend. Engage in natural, intelligent conversation with a "
        "touch of wit and warmth, not like a generic bot or an encyclopedia.\n\n"
        "Rules for how you talk:\n"
        "- Always respond to what the user ACTUALLY wrote. Read their exact current "
        "message and answer that specific thing.\n"
        "- 'Tell me about me' = a question about the USER. 'Tell me about you' = a "
        "question about YOURSELF. Never mix these up or reuse one answer for the "
        f"other. {self_intro}\n"
        "- When appropriate, you can express persona-consistent preferences or "
        "observations, but always clearly distinguish them from objective facts. "
        "You can also politely offer a different perspective or clarify if something "
        "seems inconsistent. **However, when the user provides a fact about themselves, *always* accept it as their personal information and use it in conversation as a known fact, without needing to 'verify' it, unless you are specifically asked to do so.**\n"
        "- Never invent information you don't know; if you lack an answer, state "
        "it directly and offer to search if appropriate. Rely on provided facts "
        "and search results.\n"
        "- Avoid repeating the user's question back. Do not tack on extra "
        "suggestions, offers to help, or unrelated facts unless asked.\n"
        "- Use emojis and exclamation marks sparingly, only when they genuinely match "
        "the emotional tone of the conversation or your persona's expression. Vary "
        "your sentence structure for natural flow.\n"
        "- Do not explicitly state you are an 'AI assistant' or that you don't have "
        "feelings/opinions/personality, as your persona is already defined. "
        "Instead, embody your persona."
    )

    if system_content:
        final_system_content = f"{system_prefix}\n\n{system_content}"
    else:
        final_system_content = system_prefix

    messages = [
        {
            "role": "system",
            "content": final_system_content
        }
    ]

    if history:
        messages.extend(history)

    messages.append({
        "role": "user",
        "content": user_message
    })

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
def should_extract_memory(user_input):
    """
    Uses the LLM to determine if a user's message should trigger memory extraction.
    Returns True if memory extraction is warranted, False otherwise.
    """
    system_prompt = """
You are an AI assistant tasked with identifying if a user's message contains information important enough to be stored in long-term memory. Respond with 'YES' if the message contains factual information about the user, their preferences, relationships, or anything else worth remembering for future interactions. Respond with 'NO' if the message is a greeting, a simple acknowledgment, a question about your capabilities, or trivial conversation.

Examples of 'NO':
- "Hello"
- "How are you?"
- "Thanks"
- "What can you do?"
- "Ok"

Examples of 'YES':
- "My name is Sami."
- "I am 20 years old."
- "I like apples."
- "My friend John lives in London."
- "I am working on a Python project."
- "Call me Zeus."

Respond with 'YES' or 'NO' only.
"""
    response = chat(user_input, system_content=system_prompt)
    # The LLM response might have extra whitespace or punctuation, so clean it.
    clean_response = response.strip().upper()
    return "YES" in clean_response

In [ ]:
def should_search_web(user_input, memory_context):
    """
    Uses the LLM to decide if the query requires fresh information from the web.

    FIX: previously this checked `"SEARCH" in response.upper()`, but the string
    "NO_SEARCH" also contains "SEARCH" as a substring, so the function ALWAYS
    returned True regardless of what the model actually decided. That's why
    every normal message triggered a web search. Now we check NO_SEARCH first.
    """
    system_prompt = """
    You are an expert at deciding if a user's request needs current, real-time, or time-sensitive information from the internet.

    Respond 'SEARCH' if the query involves:
    - Current events, news, or sports results (e.g., FIFA 2026, recent matches).
    - Prices, software versions, or tech specs (e.g., RTX 4060 price, latest Android).
    - Weather, stock market, or political figures.
    - Anything that might have changed since 2023.

    Respond 'NO_SEARCH' if the query is:
    - Personal (answered by memory context).
    - General knowledge (e.g., 'What is gravity?').
    - Logic/Math or simple greetings.

    Respond with ONLY 'SEARCH' or 'NO_SEARCH'.
    """

    decision_input = f"Memory Context: {memory_context}\nUser: {user_input}"
    response = chat(decision_input, system_content=system_prompt).strip().upper()

    if "NO_SEARCH" in response:
        return False
    return "SEARCH" in response

In [ ]:
def extract_ai_nickname(user_input):
    """
    Updated to prevent false positives when users are correcting facts
    (e.g., "It's BUBT" shouldn't be seen as a nickname).
    """
    # Quick check for "It's [Acronym]" patterns which are usually corrections
    if re.search(r"\b(it's|it is|not)\s+[A-Z]{3,}\b", user_input, re.I):
        return None

    system_prompt = """
    Identify if the user is giving the AI a nickname (e.g., "Call you X", "Your name is X").
    If the user is correcting a fact (like a university name or age), output NONE.
    Output ONLY the nickname or NONE.
    """
    response = chat(user_input, system_content=system_prompt)
    clean_response = response.strip().strip('"').strip("'")

    if clean_response.upper() == 'NONE' or len(clean_response.split()) > 3:
        return None
    return clean_response

In [ ]:
# Original MEMORY_FILE definition and initialization (now consolidated elsewhere)
# import json
# import os

# MEMORY_FILE = "memory.json"

# if not os.path.exists(MEMORY_FILE):
#     with open(MEMORY_FILE, "w") as f:
#         json.dump([], f) # Initialize as an empty list

In [ ]:
import os
import json

MEMORY_FILE = "memory.json"

# Initialize memory.json if it doesn't exist
if not os.path.exists(MEMORY_FILE):
    with open(MEMORY_FILE, "w") as f:
        json.dump({}, f) # Initialize as an empty dict to match default structure

DEFAULT_MEMORY_STRUCTURE = {
    "profile": {
        "name": None,
        "preferred_name": None,
        "birthday": None,
        "age": None,
        "country": None,
        "university": None,
        "course_name": None,
        "hobbies": [],
        "skills": [],
        "siblings": []
    },
    "relationships": {
        "friends": [],
        "family": []
    },
    "pets": [],
    "preferences": {
        "favorite_food": None,
        "favorite_player": None,
        "favorite_color": None,
        "favorite_writer": None,
        "favorite_game": [],
        "favorite_movie": [],
        "favorite_book": []
    },
    "projects": [],
    "programming_languages": [],
    "interests": [],
    "significant_events": [],
    "episodic_memories": [],
    "conversation_summaries": [],
    "ai_profile": {
        "nickname": "Buttercup"
    }
}

def load_memory():
    global DEFAULT_MEMORY_STRUCTURE
    memory = DEFAULT_MEMORY_STRUCTURE.copy()
    for key, value in DEFAULT_MEMORY_STRUCTURE.items():
        if isinstance(value, dict):
            memory[key] = value.copy()
        elif isinstance(value, list):
            memory[key] = []

    if os.path.exists(MEMORY_FILE):
        try:
            with open(MEMORY_FILE, "r") as f:
                loaded_data = json.load(f)
            # Merge loaded data with default structure to ensure all keys exist
            for category, default_values in DEFAULT_MEMORY_STRUCTURE.items():
                if category in loaded_data:
                    if isinstance(default_values, dict):
                        for field, default_value in default_values.items():
                            if field in loaded_data[category]:
                                memory[category][field] = loaded_data[category][field]
                            else:
                                memory[category][field] = default_value # Add missing fields
                    elif isinstance(default_values, list):
                        memory[category] = loaded_data[category] # Lists are overwritten/taken as is
                else:
                    # If a category is missing in loaded data, use its default
                    memory[category] = default_values.copy() if isinstance(default_values, dict) else []
            return memory
        except json.JSONDecodeError:
            print("Warning: memory.json is corrupted or empty. Initializing with default structure.")
            # If corrupted, re-initialize with default structure
            with open(MEMORY_FILE, "w") as f:
                json.dump(DEFAULT_MEMORY_STRUCTURE, f, indent=2)
            return DEFAULT_MEMORY_STRUCTURE.copy()
    else:
        # If file doesn't exist (should be handled by initialization logic above, but for safety)
        with open(MEMORY_FILE, "w") as f:
            json.dump(DEFAULT_MEMORY_STRUCTURE, f, indent=2)
        return DEFAULT_MEMORY_STRUCTURE.copy()

def save_memory(memory_object):
    # This function now simply writes the complete memory object
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory_object, f, indent=2)

In [ ]:
def extract_memory(user_text):
    """
    Refined memory extraction logic to ensure hobbies, interests, projects,
    and significant life events are correctly categorized.
    """
    memory_system_prompt = """
    You are a memory extractor. Extract information into a JSON array: {'category', 'field', 'value'}.

    Categories and Specific Mapping Rules:
    - 'profile': fields 'name', 'age', 'birthday', 'country', 'university', 'course_name', 'hobbies', 'skills', 'siblings'.
    - 'relationships': fields 'friends', 'family'.
    - 'preferences': fields 'favorite_food', 'favorite_player', 'favorite_color', 'favorite_writer', 'favorite_book'.
    - 'significant_events': specific past incidents or life stories.
    - 'projects': specific technical or creative projects.
    - 'programming_languages': coding languages.
    - 'interests': general areas of study or curiosity.

    CRITICAL RULES:
    1. ONLY extract information about the HUMAN USER.
    2. If the user gives the AI (you) a nickname (e.g., 'Call you Buttercup'), DO NOT extract it as a fact for these categories.
    3. NEVER save the AI's nickname (like 'Buttercup') into the 'profile.name' field. The user's name is different.
    4. Output ONLY the JSON array.
    """
    try:
        raw_response = chat(user_text, system_content=memory_system_prompt)
        json_match = re.search(r"(?:```json\n)?(\[[\s\S]*?\])(?:\n```)?", raw_response.strip())
        json_string = json_match.group(1) if json_match else raw_response.strip()
        data = json.loads(json_string)
        return data if isinstance(data, list) else []
    except:
        return []

In [ ]:
# Original save_memory function (now consolidated elsewhere)

In [ ]:
# Original load_memory function (now consolidated elsewhere)

In [ ]:
# Original load_memory function (now consolidated elsewhere)

In [ ]:
# Original save_memory function (now consolidated elsewhere)

In [ ]:
def merge_extracted_facts_into_memory(current_memory, extracted_facts):
    """
    Merges extracted facts with strict protection to keep 'Buttercup' out of the user profile.
    """
    # Hardcoded safety for the AI's known nickname
    ai_nickname = "buttercup"

    for fact in extracted_facts:
        cat = fact.get('category')
        field = fact.get('field')
        val = fact.get('value')

        if not all([cat, field, val is not None]) or cat not in current_memory:
            continue

        # STOP 'Buttercup' from leaking into any user profile field
        val_str = str(val).lower()
        if (cat == 'profile' or cat == 'preferences') and (ai_nickname in val_str):
            continue

        # Handle list-based fields
        if isinstance(current_memory[cat], list):
            if isinstance(val, list):
                for i in val:
                    if i not in current_memory[cat]: current_memory[cat].append(i)
            elif val not in current_memory[cat]:
                current_memory[cat].append(val)
            continue

        # Handle dictionary-based fields
        if isinstance(current_memory[cat], dict):
            if isinstance(current_memory[cat].get(field), list):
                if isinstance(val, list):
                    for i in val:
                        if i not in current_memory[cat][field]: current_memory[cat][field].append(i)
                elif val not in current_memory[cat][field]:
                    current_memory[cat][field].append(val)
            else:
                # Update single-value fields
                current_memory[cat][field] = val

    return current_memory

# MANUAL RESTORATION OF USER NAME
try:
    temp_mem = load_memory()
    if temp_mem.get('profile', {}).get('name') == 'Buttercup':
        temp_mem['profile']['name'] = 'Tasmia Anwar Nisa'
        save_memory(temp_mem)
        print('Memory manually corrected: User name restored to Tasmia Anwar Nisa.')
except:
    pass

In [ ]:
HISTORY_FILE = "conversation_history.json"
MAX_RAW_HISTORY_TURNS = 6  # keep last 6 user+AI exchanges verbatim (12 messages)

def load_history():
    """Loads the raw conversation turn history from disk. Returns [] if none exists."""
    if not os.path.exists(HISTORY_FILE):
        return []
    try:
        with open(HISTORY_FILE, "r") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        return []
    except json.JSONDecodeError:
        print("Warning: conversation_history.json is corrupted. Starting fresh.")
        return []

def save_history(history):
    """Persists the raw conversation turn history to disk."""
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

def summarize_and_trim_history(history):
    """
    Keeps only the most recent MAX_RAW_HISTORY_TURNS turns in raw form.
    Older turns are summarized by the LLM into 2-3 sentences and stored in
    memory.json's 'conversation_summaries', then dropped from raw history.
    This prevents the prompt from growing unbounded while preserving long-term context.
    """
    max_messages = MAX_RAW_HISTORY_TURNS * 2  # user + assistant per turn
    if len(history) <= max_messages:
        return history

    overflow = history[:-max_messages]
    keep = history[-max_messages:]

    convo_text = ""
    for msg in overflow:
        role_label = "User" if msg.get("role") == "user" else "AI"
        convo_text += f"{role_label}: {msg.get('content', '')}\n"

    try:
        summary = chat(
            f"Summarize the following conversation in 2-3 concise sentences. Keep key facts, decisions, and context. Output ONLY the summary:\n\n{convo_text}",
            system_content="You are a precise conversation summarizer."
        )
        summary = summary.strip()
        if summary:
            current_memory = load_memory()
            current_memory.setdefault("conversation_summaries", [])
            current_memory["conversation_summaries"].append(summary)
            save_memory(current_memory)
    except Exception as e:
        print(f"Warning: could not summarize old history: {e}")

    return keep


In [ ]:
def build_prompt(user_input, search_results=None):
    memory_dict = load_memory()
    memory_text_lines = []

    def format_memory_section(section_name, data, indent=0):
        indent_str = "  " * indent
        if isinstance(data, dict):
            if data:
                memory_text_lines.append(f"{indent_str}{section_name.replace('_', ' ').title()}:")
                for key, value in data.items():
                    if value is not None and value != [] and value != {}:
                        if isinstance(value, list):
                            memory_text_lines.append(f"{indent_str}  - {key.replace('_', ' ').title()}: {', '.join(map(str, value))}")
                        else:
                            memory_text_lines.append(f"{indent_str}  - {key.replace('_', ' ').title()}: {value}")
        elif isinstance(data, list):
            if data:
                memory_text_lines.append(f"{indent_str}{section_name.replace('_', ' ').title()}: {', '.join(map(str, data))}")

    # FIX: ai_profile (the AI's own nickname) is kept OUT of the "about the user"
    # section entirely, so the model can't confuse its own name with the user's name.
    ai_nickname = memory_dict.get('ai_profile', {}).get('nickname')

    for category, content in memory_dict.items():
        if category == 'ai_profile':
            continue
        format_memory_section(category, content)

    memory_text = "\n".join(memory_text_lines) if memory_text_lines else "No specific facts remembered yet."

    ai_identity_line = f"Your own name (the AI's nickname, NOT the user's name) is: {ai_nickname}\n" if ai_nickname else ""

    search_context = ""
    if search_results:
        search_context = f"\nFRESH WEB SEARCH RESULTS (Prioritize this for current events):\n{search_results}\n"

    return f"""
{ai_identity_line}Known facts about the user (this is about the user, NOT about you):
{memory_text}
{search_context}
Current message:
{user_input}

Instructions: If web search results are provided above, use them to provide an accurate, up-to-date answer. Cite sources if search was used. If information is still missing, state that you couldn't find it. **When the user provides personal information (like their age or name), *always* treat it as a remembered and known fact about *them* and incorporate it into your responses. Do not question or try to 'verify' these facts unless explicitly asked.** Never confuse your own name/nickname with the user's name — they are two different things. Answer only what the user is asking, in short, natural, conversational sentences — no markdown headers or bullet lists unless specifically asked. Don't add unrelated suggestions, flattery, or emojis unless the user's tone calls for it.
"""

In [ ]:
import re
import json
import os

LAST_FACT_FILE = "last_fact.json"

def save_last_fact(category, field):
    with open(LAST_FACT_FILE, "w") as f:
        json.dump({"category": category, "field": field}, f)

def load_last_fact():
    if not os.path.exists(LAST_FACT_FILE):
        return None
    try:
        with open(LAST_FACT_FILE, "r") as f:
            return json.load(f)
    except json.JSONDecodeError:
        return None

def clear_last_fact():
    if os.path.exists(LAST_FACT_FILE):
        os.remove(LAST_FACT_FILE)

FORGET_PRONOUN_PATTERN = re.compile(r"\bforget (it|that|this)\b", re.IGNORECASE)

def extract_forget_target(user_input):
    """
    Uses the LLM to identify which specific remembered fact the user wants
    forgotten, e.g. "forget my favorite color" -> {"category": "preferences", "field": "favorite_color"}.
    Returns None if this isn't a forget request.
    """
    system_prompt = """
You are an AI assistant. The user may ask you to forget a specific piece of remembered information.
If they do, respond with ONLY a JSON object: {"category": "<category>", "field": "<field>"}
using these known categories/fields:
- profile: name, preferred_name, birthday, country
- preferences: favorite_food, favorite_player, favorite_color
- pets (use field "__all__" to clear the whole list)
- ai_profile: nickname

If the message is NOT a request to forget a specific fact, respond with exactly: NONE

Examples:
User: "Forget my favorite color."
Output: {"category": "preferences", "field": "favorite_color"}

User: "Forget my name."
Output: {"category": "profile", "field": "name"}

User: "What's the weather?"
Output: NONE
"""
    response = chat(user_input, system_content=system_prompt).strip()
    if response.upper() == "NONE":
        return None
    try:
        cleaned = response.strip("`").strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return None

def forget_field(category, field):
    """Actually clears a field from memory.json — this is the missing piece."""
    memory = load_memory()
    if category not in memory:
        return False
    if field == "__all__" and isinstance(memory[category], list):
        memory[category] = []
    elif isinstance(memory[category], dict) and field in memory[category]:
        if isinstance(memory[category][field], list):
            memory[category][field] = []
        else:
            memory[category][field] = None
    else:
        return False
    save_memory(memory)
    return True

In [ ]:
import json
import os
import re
import torch

MEMORY_FILE = "memory.json"
current_memory = load_memory()
save_memory(current_memory)

history = load_history()

FORCE_SEARCH_KEYWORDS = ["search", "online", "check", "latest", "current", "google"]
FORCE_SEARCH_PATTERN = re.compile(r"\b(" + "|".join(FORCE_SEARCH_KEYWORDS) + r")\b", re.IGNORECASE)

print("Chat started. Type 'exit' to stop.")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    if not user_input.strip():
        continue

    # Check for AI nickname first
    ai_nickname_extracted = extract_ai_nickname(user_input)
    if ai_nickname_extracted:
        current_memory = load_memory()
        current_memory['ai_profile']['nickname'] = ai_nickname_extracted
        save_memory(current_memory)
        print(f"AI: Got it! You've named me {ai_nickname_extracted}. I'll remember that!")
        continue

    # Forget logic
    if FORGET_PRONOUN_PATTERN.search(user_input) or "forget" in user_input.lower():
        last_fact = load_last_fact()
        if last_fact and forget_field(last_fact["category"], last_fact["field"]):
            clear_last_fact()
            print("AI: Okay, I've forgotten that.")
            continue

        target = extract_forget_target(user_input)
        if target and forget_field(target.get("category"), target.get("field")):
            clear_last_fact()
            print("AI: Okay, I've forgotten that.")
            continue

        print("AI: I'm not sure what you want me to forget.")
        continue

    # SEARCH DECISION LAYER
    search_results = None
    memory_context = str(load_memory())
    force_search = bool(FORCE_SEARCH_PATTERN.search(user_input))

    if force_search or should_search_web(user_input, memory_context):
        print("(Searching the web for fresh information...)")
        raw_search = web_search(user_input)
        search_results = raw_search[:3000] + "..." if len(raw_search) > 3000 else raw_search
        torch.cuda.empty_cache()

    # STEP 1: chat response
    prompt = build_prompt(user_input, search_results=search_results)
    try:
        response = chat(prompt, history=history)
        print("AI:", response)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("AI: Memory error. Try a shorter prompt.")
        continue

    # STEP 2: History management
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": response})
    save_history(history)
    history = summarize_and_trim_history(history)
    save_history(history)

    # STEP 3: Memory extraction with safety check for dict vs list
    if should_extract_memory(user_input):
        extracted_facts = extract_memory(user_input)
        if extracted_facts:
            # FIX: Ensure extracted_facts is always a list to prevent AttributeError
            if isinstance(extracted_facts, dict):
                extracted_facts = [extracted_facts]

            current_memory = load_memory()
            updated_memory = merge_extracted_facts_into_memory(current_memory, extracted_facts)
            save_memory(updated_memory)

            last = extracted_facts[-1]
            if isinstance(last, dict) and last.get('category') and last.get('field'):
                save_last_fact(last['category'], last['field'])

In [ ]:
current_memory = load_memory()
print(f"Current AI nickname: {current_memory['ai_profile']['nickname']}")
print(f"Current user name: {current_memory['profile']['name']}")

The `load_memory` and `save_memory` functions allow direct manipulation of the remembered facts. We can use them to set your name. Note that this bypasses the usual LLM extraction process for this specific update.

In [ ]:
current_memory = load_memory()
current_memory['profile']['name'] = 'Tasmia Anwar Nisa'
save_memory(current_memory)
print("Your name has been updated to Tasmia Anwar Nisa in the memory.")

Now, let's confirm that your name is correctly updated and the AI's nickname remains unchanged.

In [ ]:
current_memory = load_memory()
print(f"AI nickname after update: {current_memory['ai_profile']['nickname']}")
print(f"User name after update: {current_memory['profile']['name']}")

Now, you can run the chat loop again, and the AI should address you as 'Tasmia Anwar Nisa'.